# GPU KV-cache experiment
Select a free GPU runtime first. This notebook installs open-source dependencies, trains on synthetic traces, and benchmarks real KV reuse. No paid API is used. Download results before the runtime expires. CUDA availability and free quotas are not guaranteed.

In [ ]:
!git clone https://github.com/swathiblrs/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference.git
%cd Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference
%pip install -e ".[inference]"

In [ ]:
import torch
assert torch.cuda.is_available(), "Select an NVIDIA GPU runtime before continuing"
print(torch.cuda.get_device_name())

In [ ]:
!python -m unittest discover -s tests -v
!kv-cache experiment --output results/experiment

## Real GPU measurements
The following downloads SmolLM2-135M weights. Synthetic token IDs test cache behavior, not language quality. A failure or output mismatch must be investigated before interpreting timings.

In [ ]:
!kv-cache benchmark --predictor results/experiment/predictor.json --device cuda --requests 100 --budget-mib 4 --new-tokens 8 --repeats 3 --output results/gpu.json

In [ ]:
import json
from pathlib import Path
report = json.loads(Path("results/gpu.json").read_text())
assert report["greedy_outputs_match"]
for run in report["runs"]:
    print(run["repeat"], run["policy"], "p95 TTFT (ms):", round(run["ttft_p95_ms"], 2), "hits:", run["hits"])

In [ ]:
import shutil
shutil.make_archive("kv-cache-results", "zip", "results")
print("Download kv-cache-results.zip using the notebook file browser.")